In [1]:
import os
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_data.csv")

print("Shape:", df.shape)
df.head()

Shape: (100000, 25)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,mort_acc,pub_rec_bankruptcies,default
0,23000.0,36 months,6.24,702.21,A,A2,1 year,RENT,150000.0,Source Verified,...,0.0,9.0,0.0,26479.0,37.2,31.0,Individual,0.0,0.0,0
1,10000.0,60 months,18.25,255.30,E,E1,< 1 year,RENT,53000.0,Source Verified,...,0.0,9.0,0.0,4784.0,52.0,18.0,Individual,0.0,0.0,1
2,25000.0,36 months,7.99,783.30,A,A5,3 years,MORTGAGE,140000.0,Source Verified,...,2.0,7.0,0.0,53593.0,15.9,16.0,Individual,2.0,0.0,0
3,5600.0,36 months,7.99,175.46,A,A4,10+ years,RENT,45000.0,Not Verified,...,1.0,12.0,0.0,4105.0,20.4,16.0,Individual,0.0,0.0,1
4,17450.0,36 months,14.99,604.83,C,C4,10+ years,RENT,48000.0,Verified,...,0.0,10.0,0.0,6554.0,72.8,17.0,Individual,0.0,0.0,0


In [ ]:
# Average FICO score
df["fico_avg"] = (
    df["fico_range_low"] + df["fico_range_high"]
) / 2

# Annual income relative to loan amount
df["income_to_loan"] = (
    df["annual_inc"] / df["loan_amnt"].replace(0, np.nan)
)

# Monthly installment relative to monthly income
df["installment_to_income"] = (
    df["installment"] /
    (df["annual_inc"].replace(0, np.nan) / 12)
)

# Convert interest rate and revolving utilization to numeric values
for col in ["int_rate", "revol_util"]:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace("%", "", regex=False),
            errors="coerce"
        )

# Convert loan term to a number of months
df["term"] = pd.to_numeric(
    df["term"]
    .astype(str)
    .str.replace(" months", "", regex=False)
    .str.strip(),
    errors="coerce"
)

# Convert employment length to years
emp = df["emp_length"].astype(str).str.strip()

df["emp_length"] = pd.to_numeric(
    emp.str.extract(r"(\d+)")[0],
    errors="coerce"
)

# Treat less than one year as zero years
df.loc[emp.str.startswith("< 1", na=False), "emp_length"] = 0

# Remove original FICO columns after creating the average
df = df.drop(
    columns=["fico_range_low", "fico_range_high"],
    errors="ignore"
)

print("Shape after feature engineering:", df.shape)
df.head()

Shape after feature engineering: (100000, 26)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,revol_bal,revol_util,total_acc,application_type,mort_acc,pub_rec_bankruptcies,default,fico_avg,income_to_loan,installment_to_income
0,23000.0,36,6.24,702.21,A,A2,1.0,RENT,150000.0,Source Verified,...,26479.0,37.2,31.0,Individual,0.0,0.0,0,717.0,6.521739,0.056177
1,10000.0,60,18.25,255.30,E,E1,0.0,RENT,53000.0,Source Verified,...,4784.0,52.0,18.0,Individual,0.0,0.0,1,662.0,5.300000,0.057804
2,25000.0,36,7.99,783.30,A,A5,3.0,MORTGAGE,140000.0,Source Verified,...,53593.0,15.9,16.0,Individual,2.0,0.0,0,767.0,5.600000,0.067140
3,5600.0,36,7.99,175.46,A,A4,10.0,RENT,45000.0,Not Verified,...,4105.0,20.4,16.0,Individual,0.0,0.0,1,687.0,8.035714,0.046789
4,17450.0,36,14.99,604.83,C,C4,10.0,RENT,48000.0,Verified,...,6554.0,72.8,17.0,Individual,0.0,0.0,0,667.0,2.750716,0.151208


In [3]:
print("Missing values by column:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nEngineered feature summary:")
print(
    df[
        [
            "fico_avg",
            "income_to_loan",
            "installment_to_income"
        ]
    ].describe()
)

Missing values by column:
emp_length               5670
mort_acc                 3771
pub_rec_bankruptcies      125
revol_util                 72
dti                        25
installment_to_income      25
delinq_2yrs                 4
inq_last_6mths              4
pub_rec                     4
open_acc                    4
total_acc                   4
verification_status         0
home_ownership              0
sub_grade                   0
installment                 0
dtype: int64

Engineered feature summary:
            fico_avg  income_to_loan  installment_to_income
count  100000.000000   100000.000000           99975.000000
mean      698.125980        7.304517               0.080968
std        31.869435        8.394449               0.396101
min       617.000000        0.000000               0.000565
25%       672.000000        3.444444               0.046182
50%       692.000000        5.000000               0.071978
75%       712.000000        8.071429               0.105273
ma

In [4]:
if "default" not in df.columns:
    raise ValueError("Target column 'default' is missing.")

y = df["default"].astype(int)

X = df.drop(
    columns=["default", "loan_status"],
    errors="ignore"
)

# Remove any columns that contain only missing values.
all_missing_cols = X.columns[X.isnull().all()].tolist()
X = X.drop(columns=all_missing_cols)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Target distribution:")
print(y.value_counts())

print("\nFeature columns:")
print(X.columns.tolist())

Features shape: (100000, 25)
Target shape: (100000,)
Target distribution:
default
0    79869
1    20131
Name: count, dtype: int64

Feature columns:
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'application_type', 'mort_acc', 'pub_rec_bankruptcies', 'fico_avg', 'income_to_loan', 'installment_to_income']


In [5]:
os.makedirs("../data/processed", exist_ok=True)

featured_data = X.copy()
featured_data["default"] = y

featured_data.to_csv(
    "../data/processed/featured_data.csv",
    index=False
)

print("Saved featured_data.csv")
print("Final shape:", featured_data.shape)

Saved featured_data.csv
Final shape: (100000, 26)
